<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/GNINA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit pandas

!wget https://github.com/gnina/gnina/releases/download/v1.0.3/gnina
!chmod +x gnina
!apt-get install openbabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 75.8 MB/s eta 0:00:00
--2026-07-16 19:35:22--  https://github.com/gnina/gnina/releases/download/v1.0.3/gnina
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/45548146/6841601b-545d-4c9f-b4b1-f0eaa8ec7b74?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-16T20%3A18%3A01Z&rscd=attachment%3B+filename%3Dgnina&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-16T19%3A17%3A10Z&ske=2026-07-16T20%3A18%3A01Z&sks=b&skv=2018-11-09&sig=Jiix%2FRBkCWurOps2OMg2uj4cKuFYMIGoWRSRa%2BAJpIk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NDIzNDEyMiwibmJmIjox

In [2]:
import pandas as pd
from rdkit import Chem

# Load predictions
df = pd.read_csv("BlindSet_UniMol_Predictions.csv")
active_ids = set(df[df["Prediction"] == "ACTIVE"]["ID"])

supplier = Chem.SDMolSupplier("BlindSet_3D.sdf", removeHs=False)

# Create two writers
writer_positive = Chem.SDWriter("Actives_Positive_For_2Na.sdf")
writer_neutral = Chem.SDWriter("Actives_Neutral_For_3Na.sdf")

# Define SMARTS patterns for physiological pH (7.4) ionization
# Matches primary, secondary, and tertiary aliphatic amines (Basic +1)
basic_pattern = Chem.MolFromSmarts('[NX3;H2,H1,H0;!$(NC=O);!$(NS=O)]')
# Matches carboxylic acids and tetrazoles (Acidic-1)
acidic_pattern = Chem.MolFromSmarts('[$([CX3](=O)[OX1H1,-]),$([nH1]c1nnnn1)]')

written_ids = set()
neutral_count = 0
positive_count = 0

print(f"Stratifying {len(active_ids)} active compounds by physiological charge...")

for mol in supplier:
    if mol is None: continue

    mol_id = mol.GetProp("_Name") if mol.HasProp("_Name") else "Unknown"

    if mol_id in active_ids and mol_id not in written_ids:

        # 1. Count Basic (+1) and Acidic (-1) groups
        num_basic = len(mol.GetSubstructMatches(basic_pattern))
        num_acidic = len(mol.GetSubstructMatches(acidic_pattern))

        # 2. Calculate Physiological Net Charge
        phys_charge = num_basic - num_acidic

        # 3. Stratify
        if phys_charge > 0:
            writer_positive.write(mol)
            positive_count += 1
        else:
            writer_neutral.write(mol)
            neutral_count += 1

        written_ids.add(mol_id)

writer_positive.close()
writer_neutral.close()


print(f"{positive_count} Cationic compounds saved for mod2.pdb (2 Na+)")
print(f"{neutral_count} Neutral compounds saved for mod3.pdb (3 Na+)")

Stratifying 58 active compounds by physiological charge...
57 Cationic compounds saved for mod2.pdb (2 Na+)
1 Neutral compounds saved for mod3.pdb (3 Na+)


In [3]:
!obabel Actives_Positive_For_2Na.sdf -O Actives_Positive_DockingReady.sdf -p 7.4
!obabel Actives_Neutral_For_3Na.sdf -O Actives_Neutral_DockingReady.sdf -p 7.4

57 molecules converted
1 molecule converted


In [4]:
!./gnina \
  --receptor mod2.pdb \
  --ligand Actives_Positive_DockingReady.sdf \
  --center_x -4.2845716 \
  --center_y 0.39099988 \
  --center_z -2.0300002 \
  --size_x 22 --size_y 22 --size_z 22 \
  --exhaustiveness 16 \
  --cnn_scoring rescore \
  --out GNINA_BlindSet_mod2.sdf

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:e9cb230+   Built Feb 11 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina --receptor mod2.pdb --ligand Actives_Positive_DockingReady.sdf --center_x -4.2845716 --center_y 0.39099988 --center_z -2.0300002 --size_x 22 --size_y 22 --size_z 22 --exhaustiveness 16 --cnn_scoring rescore --out GNINA_BlindSet_mod2.sdf
Using random seed: -854441096

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  affinity  |    CNN     |   CNN
     | (kcal/mol) | pose score | affinity
-----+------------+------------+----------
    1       -6.63       0.7411      5.003
    2       -7.26       0.6961      5

In [7]:
!./gnina \
  --receptor mod3.pdb \
  --ligand Actives_Neutral_DockingReady.sdf \
  --center_x 5.3150005 \
  --center_y -0.42442855 \
  --center_z -1.2644285 \
  --size_x 22 --size_y 22 --size_z 22 \
  --exhaustiveness 16 \
  --cnn_scoring rescore \
  --out GNINA_BlindSet_mod3.sdf

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:e9cb230+   Built Feb 11 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina --receptor mod3.pdb --ligand Actives_Neutral_DockingReady.sdf --center_x 5.3150005 --center_y -0.42442855 --center_z -1.2644285 --size_x 22 --size_y 22 --size_z 22 --exhaustiveness 16 --cnn_scoring rescore --out GNINA_BlindSet_mod3.sdf
Using random seed: 394222316

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Compound_123 | pose 0 | initial pose not within box

mode |  affinity  |    CNN     |   CNN
     | (kcal/mol) | pose score | affinity
-----+------------+------------+----------
    1       -9.59       0.454

In [8]:
import pandas as pd
from rdkit import Chem
import os

def extract_top_poses(sdf_file, model_type):
    if not os.path.exists(sdf_file):
        return pd.DataFrame()

    # sanitize=False prevents RDKit from crashing on GNINA's bond/charge formatting
    suppl = Chem.SDMolSupplier(sdf_file, sanitize=False, removeHs=False)
    results = []
    seen_ids = set()

    for mol in suppl:
        if mol is None: continue

        mol_id = mol.GetProp("_Name") if mol.HasProp("_Name") else "Unknown"

        # GNINA sorts poses automatically, so the first time we see an ID, it's the Top Pose
        if mol_id not in seen_ids:
            # Extract scores
            cnn_aff = mol.GetProp("CNN_affinity") if mol.HasProp("CNN_affinity") else "N/A"

            cnn_pose = "N/A"
            if mol.HasProp("CNN_VS"): cnn_pose = mol.GetProp("CNN_VS")
            elif mol.HasProp("CNN_pose_score"): cnn_pose = mol.GetProp("CNN_pose_score")

            vina_score = mol.GetProp("minimizedAffinity") if mol.HasProp("minimizedAffinity") else "N/A"

            results.append({
                "Compound_ID": mol_id,
                "Model": model_type,
                "CNN_Affinity": float(cnn_aff) if cnn_aff != "N/A" else cnn_aff,
                "CNN_Pose_Score": float(cnn_pose) if cnn_pose != "N/A" else cnn_pose,
                "Vina_Energy": float(vina_score) if vina_score != "N/A" else vina_score
            })
            seen_ids.add(mol_id)

    return pd.DataFrame(results)

# Extract from both files
df_pos = extract_top_poses("GNINA_BlindSet_mod2.sdf", "2 Na+ (Cationic)")
df_neu = extract_top_poses("GNINA_BlindSet_mod3.sdf", "3 Na+ (Neutral)")

# Combine and sort by the highest CNN Affinity
df_final = pd.concat([df_pos, df_neu], ignore_index=True)

if not df_final.empty:
    df_final = df_final.sort_values(by="CNN_Affinity", ascending=False)
    print(df_final.to_string(index=False))
    df_final.to_csv("Final_Docking_Scores.csv", index=False)
    print("\nSaved to Final_Docking_Scores.csv")
else:
    print("No docking results found. Check your file names!")

 Compound_ID            Model CNN_Affinity  CNN_Pose_Score  Vina_Energy
  Compound_8 2 Na+ (Cationic)          N/A        3.707738     -6.63138
Compound_138 2 Na+ (Cationic)          N/A        3.950483     -4.94735
Compound_115 2 Na+ (Cationic)          N/A        2.510717     -7.47404
Compound_116 2 Na+ (Cationic)          N/A        3.195091     -6.37313
Compound_118 2 Na+ (Cationic)          N/A        3.662584     -8.14639
Compound_119 2 Na+ (Cationic)          N/A        4.288187     -8.29928
Compound_120 2 Na+ (Cationic)          N/A        3.172981     -6.66924
Compound_122 2 Na+ (Cationic)          N/A        3.479861     -6.96996
Compound_124 2 Na+ (Cationic)          N/A        3.216858     -6.74537
Compound_125 2 Na+ (Cationic)          N/A        3.636417     -6.67284
Compound_127 2 Na+ (Cationic)          N/A        3.182202     -7.65638
Compound_128 2 Na+ (Cationic)          N/A        2.765532     -5.83861
Compound_129 2 Na+ (Cationic)          N/A        3.987473     -

[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:00:45] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol